# Run Each Generator Separately

This notebook calls `run.py` once per pipeline using `--only`. Run only the cell for the model you want to test, because some cells can use paid API calls.

## Important Notes

This notebook is only a convenience runner for the project’s main command-line entry point, `master_connections/run.py`. Each section runs one pipeline separately using `--only`, so we can test or demonstrate each contributor’s generator without running the full weighted sampler.

Some pipelines require API keys. Burak uses LLM calls for some labeling and validation steps, Adreama uses OpenAI, Kevin Fresh uses OpenAI in fresh mode, and Abuzar AI uses Anthropic/Claude. Abuzar NLP uses local datasets and does not require an API key for normal puzzle generation.

Kevin Fresh also depends on a local HuggingFace SentenceTransformer model, `all-mpnet-base-v2`. The model must be downloaded once before offline notebook runs work reliably. After that, the notebook forces HuggingFace/Transformers offline mode so Kevin does not fail because of optional network checks.

The first setup cell should be rerun whenever the notebook kernel restarts. It loads `.env`, selects the project virtual environment, creates a temporary `python` wrapper for subprocesses, and sets environment variables needed by Kevin Fresh.

The normal model cells call `run.py`, so successful outputs are routed through the master generator and can be added to the shared archive:

`master_connections/output/generated_puzzles.json`


If a model prints `Return code: 0`, that only means the command finished. It does not always mean a puzzle was accepted. Check the text above it for messages like “Generated,” “Accepted,” “Saved,” “duplicate,” “no output file found,” or “Gave up after attempts.”


In [26]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import tempfile

# Auto-detect project root by walking up from current directory.
# Works whether notebook is opened from repo root or master_connections.
start = Path.cwd().resolve()
PROJECT_DIR = None
for candidate in [start, *start.parents]:
    if (candidate / "run.py").exists():
        PROJECT_DIR = candidate
        break
    if (candidate / "master_connections" / "run.py").exists():
        PROJECT_DIR = candidate / "master_connections"
        break

assert PROJECT_DIR is not None and (PROJECT_DIR / "run.py").exists(), (
    f"Could not find master_connections/run.py from {start}"
)

def load_env_file(path):
    """Small .env loader so this notebook does not require python-dotenv."""
    path = Path(path)
    if not path.exists():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ[key] = value
    return True

env_loaded = load_env_file(PROJECT_DIR / ".env")

VENV_PYTHON = PROJECT_DIR / ".venv" / "bin" / "python"
PYTHON = str(VENV_PYTHON if VENV_PYTHON.exists() else sys.executable)

# Some adapters call subprocesses with the command name "python".
# VS Code/Jupyter kernels may not put that exact executable on PATH,
# so create a temporary shim named "python" that points to this interpreter.
python_bin = str(Path(PYTHON).parent)
shim_dir = Path(tempfile.gettempdir()) / "nyt_connections_python_shim"
shim_dir.mkdir(parents=True, exist_ok=True)
shim_python = shim_dir / "python"
if shim_python.exists() or shim_python.is_symlink():
    shim_python.unlink()
# Use a shell wrapper, not a symlink. A symlink outside .venv can make Python
# lose its virtualenv context and fail to find installed packages.
shim_python.write_text(f"#!/bin/sh\nexec {shlex.quote(PYTHON)} \"$@\"\n", encoding="utf-8")
shim_python.chmod(0o755)
os.environ["PATH"] = str(shim_dir) + os.pathsep + python_bin + os.pathsep + os.environ.get("PATH", "")

# Kevin Fresh uses a HuggingFace SentenceTransformer model. After the model is
# downloaded once, keep HF/Transformers offline so it does not fail on optional
# network checks during notebook runs.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print(f"Project dir: {PROJECT_DIR}")
print(f"Python:      {PYTHON}")
print(f"PATH first:  {shim_dir}")
print(f"Shim python: {shim_python} runs {PYTHON}")
print(f"HF offline:  {os.environ.get('HF_HUB_OFFLINE')}")
print(f".env loaded:  {env_loaded}")
print(f"OpenAI key:  {'set' if os.environ.get('OPENAI_API_KEY') else 'missing'}")

Project dir: /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections
Python:      /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python
PATH first:  /var/folders/bc/qt631mnn1gg6n6dm1lskj_780000gn/T/nyt_connections_python_shim
Shim python: /var/folders/bc/qt631mnn1gg6n6dm1lskj_780000gn/T/nyt_connections_python_shim/python runs /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python
HF offline:  1
.env loaded:  True
OpenAI key:  set


In [27]:
def run_model(pipeline, n=1, *, agent=False, dry_run=False, preview=False, check=False):
    """Run one pipeline through master_connections/run.py and stream output."""
    cmd = [PYTHON, "run.py", "--only", pipeline, "--n", str(n)]
    if agent:
        cmd.append("--agent")
    if dry_run:
        cmd.append("--dry-run")
    if preview:
        cmd.append("--preview")

    print("$", shlex.join(cmd))
    print()

    run_env = os.environ.copy()
    run_env["HF_HUB_OFFLINE"] = "1"
    run_env["TRANSFORMERS_OFFLINE"] = "1"
    if "shim_dir" in globals():
        run_env["PATH"] = str(shim_dir) + os.pathsep + run_env.get("PATH", "")

    print(f"HF_HUB_OFFLINE={run_env.get('HF_HUB_OFFLINE')}")
    print(f"TRANSFORMERS_OFFLINE={run_env.get('TRANSFORMERS_OFFLINE')}")
    print()

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=run_env,
    )

    output_lines = []
    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    returncode = process.wait()
    print(f"\nReturn code: {returncode}")

    if check and returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd, output="".join(output_lines))

    return {"pipeline": pipeline, "returncode": returncode, "output": "".join(output_lines)}

## Optional: Registration Check

This checks whether each pipeline can register. It does not generate puzzles or make API calls.

In [28]:
PIPELINES = ["burak", "adreama", "kevin_fresh", "abuzar_nlp", "abuzar_ai"]

registration_results = {}
for pipeline in PIPELINES:
    print("=" * 80)
    registration_results[pipeline] = run_model(pipeline, dry_run=True)

$ '/Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python' run.py --only burak --n 1 --dry-run

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Common words : 9,474
Full dict    : 234,375  Short: 4,622  CMU: 123,455
Utilities loaded.
Classifier loaded.
vectors.bin loaded.
Compounds: 317 prefixes, 273 suffixes.
Anagram buckets: 72 with 4+ words.
Verb-noun: 250 groups.
All 8 generators ready.
Scoring + master generator ready.
Building rhyme index...
Valid rhyme endings   : 200
Building anagram index...
Valid anagram groups  : 4
Building letter pattern index...
Valid letter patterns : 20
Registered: burak
DedupStore: 242 puzzles loaded from /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['burak']
  Weights   : {'burak': 1.0}
  Store     : 242 puzzles so far

Dry run complete (no puzzles generated).

Return 

## Burak

Runs only Burak's pipeline.

In [4]:
burak_result = run_model("burak", n=1)

$ '/Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python' run.py --only burak --n 1

Common words : 9,474
Full dict    : 234,375  Short: 4,622  CMU: 123,455
Utilities loaded.
Classifier loaded.
vectors.bin loaded.
Compounds: 317 prefixes, 273 suffixes.
Anagram buckets: 72 with 4+ words.
Verb-noun: 250 groups.
All 8 generators ready.
Scoring + master generator ready.
Building rhyme index...
Valid rhyme endings   : 200
Building anagram index...
Valid anagram groups  : 4
Building letter pattern index...
Valid letter patterns : 20
Registered: burak
DedupStore: 240 puzzles loaded from /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['burak']
  Weights   : {'burak': 1.0}
  Store     : 240 puzzles so far
  Attempt 1: [burak]...   [compound_prefix] score=1.000 (keep) ['ARM', 'CAST', 'RUNNER', 'HEAD']
  [v

## Adreama

Runs only Adreama's pipeline. Requires `OPENAI_API_KEY`.

In [5]:
adreama_result = run_model("adreama", n=1)

$ '/Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python' run.py --only adreama --n 1

Registered: adreama
DedupStore: 241 puzzles loaded from /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['adreama']
  Weights   : {'adreama': 1.0}
  Store     : 241 puzzles so far
  Attempt 1: [adreama]... Config loaded.
Memory loaded.
  Words used   : 0
  Categories   : 0
  Boards saved : 0
  Puzzles in CSV: 0
Total webs: 48
48 category webs loaded.
Generator loaded.
duplicate — duplicate purple group (words): ['MURDER', 'PARLIAMENT', 'CHARM', 'GAGGLE']
  Attempt 2: [adreama]... None (9.5s)
  Attempt 3: [adreama]... None (7.9s)
  Attempt 4: [adreama]... None (7.5s)
  Attempt 5: [adreama]... OK (1.7s) → puzzle #242
  Source  : adreama
  YELLOW : types of adhesives                       ['TAPE', 'GLUE', 'EPOXY', 

## Kevin Fresh

Runs only Kevin Fresh. Requires `OPENAI_API_KEY`.

In [35]:
kevin_fresh_result = run_model("kevin_fresh", n=1)

$ '/Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python' run.py --only kevin_fresh --n 1

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Registered: kevin_fresh
DedupStore: 245 puzzles loaded from /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['kevin_fresh']
  Weights   : {'kevin_fresh': 1.0}
  Store     : 245 puzzles so far
  Attempt 1: [kevin_fresh]... duplicate — duplicate yellow group (words): ['EXCEEDINGLY', 'WILDLY', 'EXTRAORDINARILY', 'EXTREMELY']
  Attempt 2: [kevin_fresh]... duplicate — duplicate yellow group (words): ['MUSICALLY', 'INSTRUMENTALIST', 'MUSICIAN', 'ORCHESTRAL']
  Gave up after 2 attempts.

Pipeline stats:
  burak               : 93
  abuzar_nlp          : 29
  kevin_fresh         : 11
  adreama             : 102
  abuzar_ai           : 10

Return code: 0


## Abuzar NLP

Runs only Abuzar NLP. This uses local datasets and does not require an API key.

In [31]:
abuzar_nlp_result = run_model("abuzar_nlp", n=1)

$ '/Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python' run.py --only abuzar_nlp --n 1

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Registered: abuzar_nlp
DedupStore: 243 puzzles loaded from /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['abuzar_nlp']
  Weights   : {'abuzar_nlp': 1.0}
  Store     : 243 puzzles so far
  Attempt 1: [abuzar_nlp]... OK (0.1s) → puzzle #244
  Source  : abuzar_nlp
  YELLOW : Merchants                                ['GROCER', 'JEWELER', 'RETAILER', 'SELLER']
  GREEN  : Houses                                   ['CONVENT', 'LODGE', 'MANOR', 'MANSION']
  BLUE   : Shoe Terms                               ['MOCCASIN', 'PUMP', 'SANDAL', 'SNEAKER']
  PURPLE : FORE ___                                 ['CAST', 'GROUND', 'SEE', 'SIGHT']


Pipeline stats:
  burak               : 

## Abuzar AI

Runs only Abuzar AI's standard fresh-puzzle path. Requires `ANTHROPIC_API_KEY`.

In [32]:
abuzar_ai_result = run_model("abuzar_ai", n=1)

$ '/Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python' run.py --only abuzar_ai --n 1

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Registered: abuzar_ai
DedupStore: 244 puzzles loaded from /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['abuzar_ai']
  Weights   : {'abuzar_ai': 1.0}
  Store     : 244 puzzles so far
  Attempt 1: [abuzar_ai]... OK (85.3s) → puzzle #245
  Source  : abuzar_ai
  YELLOW : Types of Dance                           ['WALTZ', 'TANGO', 'SALSA', 'FOXTROT']
  GREEN  : Add 'LE' to the Front to Make a New Word ['GACY', 'AGUE', 'APING', 'GEND']
  BLUE   : Words That Can Precede 'WORK'            ['FRAME', 'GROUND', 'GUESS', 'CLOCK']
  PURPLE : Common Words That Are Also Weaving Terms ['WARP', 'WEFT', 'REED', 'SHUTTLE']


Pipeline stats:
  burak               : 93
  abuzar_nlp     

## Optional: Abuzar AI Agentic Mode

Runs Abuzar AI through `run.py --agent`. This can be slower and can use more API calls. Note: This model is expensive ($0.2 - $1 per call) to run, we suggest running it only a few times to test.

In [34]:
abuzar_ai_agent_result = run_model("abuzar_ai", n=1, agent=True)

$ '/Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python' run.py --only abuzar_ai --n 1 --agent

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Registered: abuzar_ai (agentic mode)
DedupStore: 245 puzzles loaded from /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['abuzar_ai']
  Weights   : {'abuzar_ai': 1.0}
  Store     : 245 puzzles so far
  Attempt 1: [abuzar_ai]...   [abuzar_ai] subprocess failed (1): Starting bank size: 68
Target new puzzles: 1
Safety limits: max_attempts=1, max_empty_batches=1, max_review_rounds=1, strategy=standard

Batch 1: requesting 1 accepted puzzle(s)
Accepted this batch: 0
Rejected this batch: 2
Bank size: 68

Stopped after 1 consecutive empty batch(es) to avoid extra API spend.

Stopped after 1 batches with 0/1 new puzzles.
None (44.8s)
  Attempt 2: [abuzar_ai]...   [abuz